In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\Landl\Downloads\Data-Science-Projects\sec-financial-nlp


In [2]:
import pickle
from bs4 import BeautifulSoup
import re
import torch
import pandas as pd
from transformers import pipeline

with open("data/processed/all_texts.pkl", "rb") as f:
    all_texts_fixed = pickle.load(f)

print(f"Loaded: {len(all_texts_fixed)} filings")
print(f"GPU available: {torch.cuda.is_available()}")

Loaded: 115 filings
GPU available: True


In [3]:
def extract_8k_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    documents = content.split("<DOCUMENT>")
    
    target_doc = None
    for doc in documents:
        if doc.strip().startswith("<TYPE>EX-99.1"):
            target_doc = doc
            break
    
    if target_doc is None:
        for doc in documents:
            if doc.strip().startswith("<TYPE>8-K"):
                target_doc = doc
                break
    
    if target_doc is None or len(target_doc) < 200:
        return ""
    
    soup = BeautifulSoup(target_doc, "html.parser")
    for tag in soup.find_all(re.compile(r'^ix:')):
        tag.decompose()
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    return text[200:] if len(text) > 200 else text


def extract_ceo_quote(text):
    sentences = text.replace('\n', ' ').split('.')
    
    patterns = [
        r'said\s+\w+.*?(?:CEO|Chief Executive|President|Chairman)',
        r'said\s+\w+.*?(?:CFO|Chief Financial)'
    ]
    
    for i, sentence in enumerate(sentences):
        for pattern in patterns:
            if re.search(pattern, sentence, re.IGNORECASE):
                start = max(0, i-3)
                quote_block = '. '.join(sentences[start:i+1])
                if len(quote_block) > 50:
                    return quote_block
    
    return ' '.join(text.split()[:300])

In [4]:
for ticker in ["AAPL", "MSFT", "GOOGL", "JPM", "TSLA"]:
    folder = f"data/raw/sec-edgar-filings/{ticker}/8-K"
    if not os.path.exists(folder):
        continue
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".txt"):
                filepath = os.path.join(root, file)
                key = f"{ticker}_8-K_{os.path.basename(root)}"
                try:
                    text = extract_8k_text(filepath)
                    if len(text) > 500:
                        all_texts_fixed[key] = text
                    else:
                        all_texts_fixed.pop(key, None)
                except Exception as e:
                    print(f"✗ {key} — {e}")

print(f"Total filings: {len(all_texts_fixed)}")

with open("data/processed/all_texts.pkl", "wb") as f:
    pickle.dump(all_texts_fixed, f)

print("Saved.")

Total filings: 125
Saved.


In [5]:
sample_key = [k for k in all_texts_fixed if "AAPL_8-K" in k][0]
print(all_texts_fixed[sample_key][:400])

ll-time high for all major product categories CUPERTINO, California — July 28, 2022 — Apple ® today announced financial results for its fiscal 2022 third quarter ended June 25, 2022. The Company posted a June quarter revenue record of $83.0 billion, up 2 percent year over year, and quarterly earnings per diluted share of $1.20. “This quarter’s record results speak to Apple’s constant efforts to in


In [6]:
sentiment_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0
)
print("FinBERT loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT loaded.


In [7]:
eightk_keys = [k for k in all_texts_fixed if "8-K" in k]

records = []
skipped = 0

for key in eightk_keys:
    text = all_texts_fixed[key]
    quote = extract_ceo_quote(text)
    
    if len(quote) < 50:
        skipped += 1
        continue
    
    truncated = " ".join(quote.split()[:400])
    result = sentiment_pipeline(truncated, truncation=True, max_length=512)
    
    records.append({
        "key": key,
        "ticker": key.split("_")[0],
        "label": result[0]["label"],
        "score": round(result[0]["score"], 4),
        "quote": quote[:200]
    })

df_sentiment = pd.DataFrame(records)
print(f"Analyzed: {len(records)} filings, Skipped: {skipped}")
print(df_sentiment["label"].value_counts())
print(df_sentiment.groupby("ticker")["label"].value_counts())

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Analyzed: 100 filings, Skipped: 0
label
neutral     64
positive    26
negative    10
Name: count, dtype: int64
ticker  label   
AAPL    neutral      9
        negative     6
        positive     5
GOOGL   neutral     15
        positive     4
        negative     1
JPM     neutral     16
        positive     4
MSFT    neutral     12
        positive     7
        negative     1
TSLA    neutral     12
        positive     6
        negative     2
Name: count, dtype: int64


YES!